# Human M1 Cell Type Classification — TOSICA-Inspired Transformer

**Dataset:** Allen Brain Atlas — Human Primary Motor Cortex (M1) 10x Chromium  
**Task:** Predict cell type from single-cell RNA-seq gene expression

## What Changed vs. the Original Notebook

| Shortcoming in Original | Fix Applied Here |
|---|---|
| 1D CNN — no sequence meaning for gene data | Transformer with multi-head self-attention |
| Only 200 HVGs selected by variance — no biological grounding | Pathway-masked token embedding (MSigDB C2 Reactome gene sets) |
| Latent space uninterpretable (abstract CNN features) | CLS→pathway attention scores = interpretable cell embeddings |
| UMAP on PCA of gene expression | UMAP on attention scores (biologically meaningful axes) |
| No 'unknown' cell handling on transfer | Probability threshold flags low-confidence / novel cells |
| Batch effect handled implicitly or not at all | Gene→pathway mapping suppresses technical noise |
| No trajectory / pathway-level analysis | Differential attention scores per cell type reveal key pathways |
| No per-pathway interpretability plots | Top-attention pathways plotted per cell type |

## Pipeline
1. Install & import dependencies  
2. Load data (same Allen Brain M1 data)  
3. Train / Val / Test split (before preprocessing — same as original)  
4. Build **pathway mask** from MSigDB Reactome gene sets  
5. Preprocess: library-size norm + log1p (no HVG selection — pathways handle filtering)  
6. **TOSICA-style Transformer** with pathway-masked embedding + CLS token  
7. Train with class-weighted cross-entropy + cosine LR  
8. Evaluate: accuracy, F1, confusion matrix  
9. **Interpretability:** attention score UMAP, top pathway heatmap, per-class pathway rankings  
10. **Cross-dataset transfer** to SMART-seq with 'Unknown' detection  

## 0. Install Dependencies

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

packages = ['torch', 'torchvision', 'umap-learn', 'tqdm', 'requests', 'scanpy', 'polars']
for pkg in packages:
    try:
        __import__(pkg.replace('-', '_').split('[')[0])
        print(f'  {pkg}: already installed')
    except ImportError:
        print(f'  Installing {pkg}...')
        install(pkg)
        print(f'  {pkg}: installed')

print('All dependencies ready.')

## 1. Imports

In [ ]:
import os
import gc
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
import math

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, balanced_accuracy_score

import umap

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import polars as pl

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## 2. Mount Drive & Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

COLAB_DATA_DIR = '/content/drive/MyDrive/FinalProject/data'
PATHS = {
    'matrix':   os.path.join(COLAB_DATA_DIR, 'matrix.csv'),
    'metadata': os.path.join(COLAB_DATA_DIR, 'metadata.csv'),
}

print('Loading metadata...')
meta = pd.read_csv(PATHS['metadata'])
print(f'Metadata shape: {meta.shape}')

print('Loading expression matrix (this may take a moment)...')
expr_raw = pl.read_csv(PATHS['matrix'], infer_schema_length=1000).to_pandas()
if 'sample_name' in expr_raw.columns:
    expr_raw.set_index('sample_name', inplace=True)
expr_raw.fillna(0, inplace=True)

label_col = 'subclass_label'
id_col    = 'sample_name'

meta_indexed = meta.set_index(id_col)
common_ids   = expr_raw.index.intersection(meta_indexed.index)
expr         = expr_raw.loc[common_ids]
labels_series = meta_indexed.loc[common_ids, label_col]

valid_mask    = labels_series.notna()
expr          = expr[valid_mask]
labels_series = labels_series[valid_mask]

all_gene_names = expr.columns.tolist()
print(f'\nFinal dataset: {expr.shape[0]:,} cells x {expr.shape[1]:,} genes')
print(f'Cell types: {labels_series.nunique()}')
print(labels_series.value_counts())

## 3. Train / Val / Test Split

In [ ]:
min_cells_per_class = 100
class_counts  = labels_series.value_counts()
valid_classes = class_counts[class_counts >= min_cells_per_class].index

filtered_mask   = labels_series.isin(valid_classes)
expr_filtered   = expr[filtered_mask]
labels_filtered = labels_series[filtered_mask]

le = LabelEncoder()
y_all = le.fit_transform(labels_filtered.values)
class_names = le.classes_
n_classes   = len(class_names)
print(f'Retained {n_classes} cell types across {expr_filtered.shape[0]:,} cells')

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.80, 0.10, 0.10

X_train_raw, X_temp, y_train, y_temp = train_test_split(
    expr_filtered.values, y_all,
    test_size=VAL_FRAC + TEST_FRAC, stratify=y_all, random_state=SEED)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=y_temp, random_state=SEED)

print(f'Train: {X_train_raw.shape[0]:,}  |  Val: {X_val_raw.shape[0]:,}  |  Test: {X_test_raw.shape[0]:,}')

## 4. Pathway Mask — The Key TOSICA Innovation

Instead of selecting 200 HVGs by variance (arbitrary, biologically blind), we build a **binary pathway mask** M where M[gene, pathway]=1 iff the gene belongs to that pathway.

- Source: MSigDB C2 Reactome gene sets (downloaded on the fly as a GMT file)
- Each pathway becomes a **token** in the Transformer
- The embedding layer is *sparse* — genes only project into their own pathways
- This means every attention score maps back to a real biological process

> TOSICA paper: *"By connecting attention to prior biological knowledge... TOSICA interpretably integrates and annotates single-cell data"*

In [ ]:
# ── Download MSigDB Reactome gene sets (GMT format) ──────────────────────────
# If you have a local copy, set GMT_PATH directly.
GMT_URL  = 'https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/c2.cp.reactome.v2023.2.Hs.symbols.gmt'
GMT_PATH = '/content/reactome.gmt'

if not os.path.exists(GMT_PATH):
    print('Downloading Reactome GMT...')
    r = requests.get(GMT_URL, timeout=120)
    with open(GMT_PATH, 'wb') as f:
        f.write(r.content)
    print(f'Saved to {GMT_PATH}')
else:
    print(f'Using cached {GMT_PATH}')

# ── Parse GMT
def parse_gmt(path, max_gene_set_size=300):
    """Returns dict: {pathway_name: [gene, ...]}"""
    gmt = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            name  = parts[0]
            genes = [g for g in parts[2:] if g]   # skip URL in column 1
            if len(genes) <= max_gene_set_size:
                gmt[name] = genes
    return gmt

gmt = parse_gmt(GMT_PATH, max_gene_set_size=300)
print(f'Gene sets loaded: {len(gmt):,}')

# Build mask matrix
MAX_PATHWAYS = 300
gene_set_universe = set(all_gene_names)

# Keep pathways that have ≥5 genes found in our dataset
valid_pathways = [
    (name, [g for g in genes if g in gene_set_universe])
    for name, genes in gmt.items()
    if len([g for g in genes if g in gene_set_universe]) >= 5
]
# Sort by coverage descending, take top MAX_PATHWAYS
valid_pathways.sort(key=lambda x: len(x[1]), reverse=True)
valid_pathways = valid_pathways[:MAX_PATHWAYS]

pathway_names = [p[0] for p in valid_pathways]
pathway_genes = [p[1] for p in valid_pathways]
n_pathways    = len(pathway_names)
print(f'Pathways used: {n_pathways}  (after filtering for overlap with dataset genes)')

# Gene universe = all genes that appear in at least one selected pathway
pathway_gene_universe = sorted(set(g for gs in pathway_genes for g in gs))
gene_to_idx = {g: i for i, g in enumerate(pathway_gene_universe)}
n_genes_pathway = len(pathway_gene_universe)
print(f'Genes covered by pathways: {n_genes_pathway:,} (of {len(all_gene_names):,} total)')

# M[gene_idx, pathway_idx] = 1 if gene in pathway
M = np.zeros((n_genes_pathway, n_pathways), dtype=np.float32)
for j, genes in enumerate(pathway_genes):
    for g in genes:
        if g in gene_to_idx:
            M[gene_to_idx[g], j] = 1.0

MASK = torch.from_numpy(M)   # shape: (n_genes_pathway, n_pathways)
print(f'Mask matrix shape: {MASK.shape}  (sparsity: {1 - M.mean():.2%})')

# Save pathway universe index for preprocessing
pathway_gene_col_indices = [
    all_gene_names.index(g) for g in pathway_gene_universe if g in all_gene_names
]
# (some genes may not be in all_gene_names if the dataset was filtered)
pathway_gene_col_indices_valid = []
pathway_gene_names_valid = []
for g in pathway_gene_universe:
    if g in all_gene_names:
        pathway_gene_col_indices_valid.append(all_gene_names.index(g))
        pathway_gene_names_valid.append(g)

# Rebuild gene_to_idx to only include genes actually in the expression matrix
pathway_gene_universe = pathway_gene_names_valid
gene_to_idx = {g: i for i, g in enumerate(pathway_gene_universe)}
n_genes_pathway = len(pathway_gene_universe)

# Rebuild mask with valid genes only
M = np.zeros((n_genes_pathway, n_pathways), dtype=np.float32)
for j, genes in enumerate(pathway_genes):
    for g in genes:
        if g in gene_to_idx:
            M[gene_to_idx[g], j] = 1.0

MASK = torch.from_numpy(M)
print(f'Final mask: {MASK.shape} — {n_genes_pathway:,} genes x {n_pathways} pathways')

## 5. Preprocessing

Instead of selecting 200 HVGs, we:
1. Select only the genes in the pathway universe (biologically grounded)
2. Library-size normalize + log1p (per cell)
3. NO StandardScaler — the pathway mask keeps features in a biologically interpretable range

Parameters are determined on the validation set (same data-leakage discipline as original).

In [ ]:
def extract_pathway_genes(X_raw, col_indices):
    """Extract pathway-relevant genes and library-size normalize + log1p."""
    X_filt = X_raw[:, col_indices].astype(np.float32)
    lib    = X_filt.sum(axis=1, keepdims=True)
    lib    = np.where(lib == 0, 1, lib)
    return np.log1p((X_filt / lib) * 1e4)

col_idx = pathway_gene_col_indices_valid   # indices into all_gene_names

X_train = extract_pathway_genes(X_train_raw, col_idx)
X_val   = extract_pathway_genes(X_val_raw,   col_idx)
X_test  = extract_pathway_genes(X_test_raw,  col_idx)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}')
print(f'Each cell is represented by {X_train.shape[1]:,} pathway-relevant genes')
del X_train_raw, X_val_raw, X_test_raw; gc.collect()

## 6. TOSICA-Style Transformer Architecture

### Architecture components:

1. **MaskedLinear (Cell Embedding)** — a fully-connected layer where the weight matrix is element-wise multiplied by the binary pathway mask M. Each output neuron receives input **only** from its pathway's genes. This is repeated `embed_dim` times in parallel.

2. **CLS Token** — a learnable parameter appended to the pathway token sequence. At inference, the CLS token aggregates all pathway information through self-attention.

3. **Multi-Head Self-Attention** — standard Transformer encoder. The attention weights between CLS and each pathway token = the **pathway importance score** for that cell.

4. **Classifier** — a small MLP on the CLS output → softmax over cell types.

5. **Attention score extraction** — the CLS→pathway attention weights are saved for UMAP and interpretability analysis.

In [ ]:
class MaskedEmbedding(nn.Module):
    """
    TOSICA Cell Embedding Layer.
    Projects genes -> pathway tokens using a sparse mask.
    W_effective = W * M  (only genes in a pathway contribute to its token)
    Repeated `embed_dim` times in parallel, then concatenated.
    """
    def __init__(self, n_genes: int, n_pathways: int, embed_dim: int, mask: torch.Tensor):
        super().__init__()
        self.n_genes     = n_genes
        self.n_pathways  = n_pathways
        self.embed_dim   = embed_dim
        # Register mask as buffer (not a parameter, but moves with .to(device))
        self.register_buffer('mask', mask)  # (n_genes, n_pathways)
        # Weight matrix — same shape as mask, repeated embed_dim times
        self.weight = nn.Parameter(torch.randn(embed_dim, n_genes, n_pathways) * 0.01)
        self.bias   = nn.Parameter(torch.zeros(embed_dim, n_pathways))

    def forward(self, x):
        # x: (batch, n_genes)
        # masked weight: (embed_dim, n_genes, n_pathways)
        W = self.weight * self.mask.unsqueeze(0)   # broadcast mask
        # x: (batch, n_genes) -> (batch, embed_dim, n_pathways)
        out = torch.einsum('bg,egp->bep', x, W) + self.bias
        return out   # (batch, embed_dim, n_pathways)


class TOSICA(nn.Module):
    """
    Transformer for One-Stop Interpretable Cell-type Annotation.
    Inspired by Chen et al., Nature Communications 2023.

    Args:
        n_genes      : number of input genes (pathway-filtered)
        n_pathways   : number of pathway tokens
        n_classes    : number of cell types
        mask         : binary mask (n_genes, n_pathways)
        embed_dim    : parallel projections per pathway (TOSICA default: 48)
        n_heads      : attention heads
        n_layers     : transformer encoder layers
        dropout      : dropout probability
    """
    def __init__(self, n_genes, n_pathways, n_classes, mask,
                 embed_dim=48, n_heads=4, n_layers=2, dropout=0.1,
                 unknown_threshold=0.95):
        super().__init__()
        self.n_pathways        = n_pathways
        self.embed_dim         = embed_dim
        self.unknown_threshold = unknown_threshold

        # Cell Embedding: genes -> pathway tokens
        self.embedding = MaskedEmbedding(n_genes, n_pathways, embed_dim, mask)

        # CLS token: shape (1, embed_dim, 1) — will be prepended to pathway tokens
        self.cls_token = nn.Parameter(torch.randn(1, embed_dim, 1))

        # Positional encoding (simple learned)
        self.pos_embed = nn.Parameter(torch.randn(1, embed_dim, n_pathways + 1) * 0.01)

        # Transformer encoder
        # Token dimension = embed_dim; sequence length = n_pathways + 1 (incl. CLS)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,   # Pre-LN for stability
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Attention extraction hook (for interpretability)
        self._attn_weights = None

        # Cell-type classifier on CLS output
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, n_classes),
        )

    def forward(self, x, return_attention=False):
        """
        x: (batch, n_genes)
        Returns logits (batch, n_classes) and optionally attention scores.
        """
        batch = x.size(0)

        # 1. Pathway token embedding: (batch, embed_dim, n_pathways)
        tokens = self.embedding(x)

        # 2. Prepend CLS token: (batch, embed_dim, 1+n_pathways)
        cls = self.cls_token.expand(batch, -1, -1)
        tokens = torch.cat([cls, tokens], dim=2)   # (batch, embed_dim, 1+n_pathways)

        # 3. Add positional embedding
        tokens = tokens + self.pos_embed

        # 4. Transpose for Transformer: (batch, seq_len, embed_dim)
        #    seq_len = 1 + n_pathways
        tokens = tokens.permute(0, 2, 1)  # (batch, 1+n_pathways, embed_dim)

        # 5. Transformer encoder
        out = self.transformer(tokens)    # (batch, 1+n_pathways, embed_dim)

        # 6. Extract CLS output (index 0)
        cls_out = out[:, 0, :]            # (batch, embed_dim)

        # 7. Classify
        logits = self.classifier(cls_out)

        if return_attention:
            # Compute CLS-to-pathway attention scores for interpretability
            # Use the last layer's attention by doing a manual QK computation
            # on the final output. This approximates TOSICA's attention embedding.
            # We take the raw pathway token outputs (indices 1:) for each cell
            # and correlate with the CLS embedding via dot-product.
            pathway_out = out[:, 1:, :]  # (batch, n_pathways, embed_dim)
            cls_rep     = cls_out.unsqueeze(1)  # (batch, 1, embed_dim)
            attn_scores = torch.bmm(cls_rep, pathway_out.transpose(1, 2))  # (batch, 1, n_pathways)
            attn_scores = F.softmax(attn_scores.squeeze(1) / math.sqrt(self.embed_dim), dim=-1)
            return logits, attn_scores  # attn_scores: (batch, n_pathways)

        return logits

    def predict_with_unknown(self, x, threshold=None):
        """Returns predicted class indices; cells below threshold -> -1 (Unknown)."""
        if threshold is None:
            threshold = self.unknown_threshold
        logits = self.forward(x)
        probs  = F.softmax(logits, dim=-1)
        max_p, preds = probs.max(dim=-1)
        preds[max_p < threshold] = -1   # flag unknowns
        return preds, max_p


# ── Instantiate model ──────────────────────────────────────────────────────────
model = TOSICA(
    n_genes    = n_genes_pathway,
    n_pathways = n_pathways,
    n_classes  = n_classes,
    mask       = MASK,
    embed_dim  = 48,   # TOSICA paper default
    n_heads    = 4,
    n_layers   = 2,
    dropout    = 0.1,
    unknown_threshold = 0.95
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,} total | {trainable_params:,} trainable')
print(model)

## 7. Dataset & DataLoaders

In [ ]:
class GeneExpressionDataset(Dataset):
    """Single-cell RNA-seq dataset. Returns (gene_vector, label)."""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = torch.from_numpy(y).long()

    def __len__(self):  return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 128

train_ds = GeneExpressionDataset(X_train, y_train)
val_ds   = GeneExpressionDataset(X_val,   y_val)
test_ds  = GeneExpressionDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Val: {len(val_loader)}  |  Test: {len(test_loader)}')
xb, yb = next(iter(train_loader))
print(f'Batch X: {xb.shape}  |  y: {yb.shape}')

## 8. Training

In [ ]:
# Class-weighted loss (handles imbalance, same as original)
class_counts_train = np.bincount(y_train, minlength=n_classes).astype(np.float32)
class_weights = torch.tensor(1.0 / (class_counts_train + 1e-6)).to(DEVICE)
class_weights = class_weights / class_weights.sum() * n_classes

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

EPOCHS       = 100
LR           = 3e-4
WEIGHT_DECAY = 1e-5
optimizer    = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler    = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_acc  = 0.0
best_val_loss = float('inf')
best_ckpt     = 'best_tosica_model.pt'
patience      = 15
no_improve    = 0
history       = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if train:
                optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct    += (logits.argmax(1) == yb).sum().item()
            total      += len(yb)
    return total_loss / total, correct / total


print(f'Training TOSICA for up to {EPOCHS} epochs on {DEVICE}...')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>9} | {"Val Acc":>8}')
print('-' * 62)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    flag = ''
    if vl_loss < best_val_loss - 1e-4:
        best_val_loss, best_val_acc = vl_loss, vl_acc
        torch.save(model.state_dict(), best_ckpt)
        no_improve = 0
        flag = ' ◀ best'
    else:
        no_improve += 1

    print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.4f} | {vl_loss:>9.4f} | {vl_acc:>8.4f}{flag}')

    if no_improve >= patience:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest val accuracy: {best_val_acc:.4f}')

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ep = range(1, len(history['train_loss']) + 1)
axes[0].plot(ep, history['train_loss'], label='Train')
axes[0].plot(ep, history['val_loss'],   label='Val')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss Curves'); axes[0].legend()
axes[1].plot(ep, history['train_acc'], label='Train')
axes[1].plot(ep, history['val_acc'],   label='Val')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Accuracy Curves'); axes[1].legend()
plt.tight_layout()
plt.savefig('fig_training_curves.png', dpi=150)
plt.show()

## 9. Test Set Evaluation

In [ ]:
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)
test_acc  = (all_preds == all_true).mean()

print(f'Test Accuracy:         {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Balanced Accuracy:     {balanced_accuracy_score(all_true, all_preds):.4f}')
print(f'Macro F1:              {f1_score(all_true, all_preds, average="macro"):.4f}')
print()
print(classification_report(all_true, all_preds, target_names=class_names, digits=3))

# Confusion matrix
cm = confusion_matrix(all_true, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(max(10, n_classes), max(8, n_classes - 2)))
ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(
    ax=ax, cmap='Blues', colorbar=True, xticks_rotation=45)
ax.set_title(f'Normalized Confusion Matrix — Test Set (Acc: {test_acc:.3f})')
plt.tight_layout()
plt.savefig('fig_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-class F1 bar chart
f1_per_class = f1_score(all_true, all_preds, average=None)
palette = sns.color_palette('tab20', n_classes)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(n_classes), f1_per_class, color=palette)
ax.axhline(f1_per_class.mean(), color='red', linestyle='--', label=f'Mean: {f1_per_class.mean():.3f}')
ax.set_xticks(range(n_classes)); ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
ax.set(ylabel='F1 Score', title='Per-Class F1 — Test Set', ylim=(0, 1.05))
ax.legend()
plt.tight_layout()
plt.savefig('fig_f1_per_class.png', dpi=150)
plt.show()

## 10. Interpretability — Attention Score Analysis

This is the core novelty vs. the original CNN. Here we:
1. Extract **CLS→pathway attention scores** for every test cell
2. Run **UMAP on attention scores** (not gene expression) — axes now represent biological pathways
3. Plot **top-attended pathways per cell type** — shows which biological processes drive each cell identity
4. Plot **pathway attention heatmap** across cell types

> TOSICA paper: *"Attention scores between CLS and pathway tokens, used as the attention embedding of cells, enable a variety of downstream analyses."*

In [ ]:
# ── Extract attention scores for the full test set ────────────────────────────
model.eval()
all_attn = []
all_true_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        _, attn = model(xb, return_attention=True)
        all_attn.append(attn.cpu().numpy())
        all_true_labels.extend(yb.numpy())

attn_matrix  = np.vstack(all_attn)          # (n_test, n_pathways)
true_labels  = np.array(all_true_labels)
true_names   = le.inverse_transform(true_labels)

print(f'Attention matrix shape: {attn_matrix.shape}')
print(f'Each row = one cell; each column = one Reactome pathway')

In [ ]:
# ── UMAP on attention scores (not gene expression!) ───────────────────────────
MAX_CELLS = 3000
if len(attn_matrix) > MAX_CELLS:
    idx = np.random.choice(len(attn_matrix), MAX_CELLS, replace=False)
    attn_sub = attn_matrix[idx]
    labs_sub  = true_names[idx]
else:
    attn_sub, labs_sub = attn_matrix, true_names

print(f'Running UMAP on attention scores ({attn_sub.shape})...')
reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3, random_state=SEED, metric='cosine')
umap_attn = reducer.fit_transform(attn_sub)

palette = {cls: sns.color_palette('tab20', n_classes)[i] for i, cls in enumerate(class_names)}

fig, ax = plt.subplots(figsize=(10, 8))
for cls in class_names:
    mask = labs_sub == cls
    if mask.sum() == 0: continue
    ax.scatter(umap_attn[mask, 0], umap_attn[mask, 1],
               c=[palette[cls]], s=3, alpha=0.7, label=cls)
ax.set(xlabel='UMAP 1', ylabel='UMAP 2',
       title='UMAP of TOSICA Attention Scores (Test Set)\n(axes = biological pathway space)')
ax.legend(markerscale=4, fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('fig_attention_umap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_attention_umap.png')
print('\n>>> Compare this UMAP to the original gene-expression UMAP.')
print('    Clusters here represent shared BIOLOGICAL PATHWAYS, not just correlated gene counts.')

In [ ]:
# ── Top-attended pathways per cell type ───────────────────────────────────────
# For each cell type, compute the mean attention score across cells, then rank pathways
mean_attn_per_class = {}
for i, cls in enumerate(class_names):
    mask = true_labels == i
    if mask.sum() == 0: continue
    mean_attn_per_class[cls] = attn_matrix[mask].mean(axis=0)  # (n_pathways,)

TOP_N = 10   # top pathways per cell type

# Clean pathway names for display
def clean_name(name, max_len=40):
    name = name.replace('REACTOME_', '').replace('_', ' ').title()
    return name[:max_len] + '...' if len(name) > max_len else name

short_pathway_names = [clean_name(p) for p in pathway_names]

fig, axes = plt.subplots(math.ceil(n_classes / 3), 3,
                          figsize=(18, 4 * math.ceil(n_classes / 3)))
axes = axes.flatten()

for i, cls in enumerate(class_names):
    if cls not in mean_attn_per_class: continue
    scores = mean_attn_per_class[cls]
    top_idx = np.argsort(scores)[::-1][:TOP_N]
    top_names = [short_pathway_names[j] for j in top_idx]
    top_scores = scores[top_idx]

    axes[i].barh(range(TOP_N), top_scores[::-1], color=palette[cls])
    axes[i].set_yticks(range(TOP_N))
    axes[i].set_yticklabels(top_names[::-1], fontsize=7)
    axes[i].set_title(cls, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Mean Attention Score', fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'Top {TOP_N} Attended Reactome Pathways per Cell Type\n(TOSICA Interpretability)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig_top_pathways_per_celltype.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_top_pathways_per_celltype.png')

In [ ]:
# ── Pathway Attention Heatmap (cell types × top pathways) ────────────────────
# Find the union of top-5 pathways per class
TOP_PER_CLASS = 5
top_pathway_set = set()
for cls, scores in mean_attn_per_class.items():
    top_idx = np.argsort(scores)[::-1][:TOP_PER_CLASS]
    top_pathway_set.update(top_idx.tolist())

top_pathway_idx = sorted(top_pathway_set)
heatmap_data = np.array([
    mean_attn_per_class[cls][top_pathway_idx] for cls in class_names
])
# Normalize per row for visibility
heatmap_norm = heatmap_data / (heatmap_data.max(axis=1, keepdims=True) + 1e-8)

top_names_display = [short_pathway_names[j] for j in top_pathway_idx]

fig, ax = plt.subplots(figsize=(max(14, len(top_pathway_idx) * 0.4), max(6, n_classes * 0.5)))
sns.heatmap(heatmap_norm, xticklabels=top_names_display, yticklabels=class_names,
            cmap='YlOrRd', ax=ax, linewidths=0.3, linecolor='lightgray')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9)
ax.set_title('Pathway Attention Heatmap — Cell Types × Reactome Pathways\n(row-normalized mean attention scores)', fontsize=11)
plt.tight_layout()
plt.savefig('fig_pathway_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_pathway_heatmap.png')
print('\n>>> Each row = one cell type; each column = one biological pathway.')
print('    Bright cells = this pathway strongly drives that cell type identity.')

## 11. Cross-Dataset Transfer with 'Unknown' Cell Detection

The original notebook transfers to SMART-seq but has no mechanism for novel cells — it forces all cells into one of the 9 training classes.

TOSICA's fix: if `max(softmax probabilities) < threshold`, label the cell **'Unknown'**. This enables discovery of novel or rare cell types not present in the reference.

> TOSICA paper: *"If the highest probability is below a preset cutoff (0.95), this cell is annotated as 'Unknown'"*

In [ ]:
NEW_MATRIX_PATH = '/content/drive/MyDrive/FinalProject/HumanMultipleCorticalAreasSMARTseq_matrix.csv'
NEW_META_PATH   = '/content/drive/MyDrive/FinalProject/HumanMultipleCorticalAreasSMARTseq_meta.csv'

# ── Load only the genes we need (memory efficient) ───────────────────────────
print('Reading CSV header...')
new_header = pl.read_csv(NEW_MATRIX_PATH, n_rows=0).columns
cols_to_load = (['sample_name'] if 'sample_name' in new_header else []) + \
               [g for g in pathway_gene_universe if g in new_header]
missing = [g for g in pathway_gene_universe if g not in new_header]
print(f'Pathway genes found: {len(cols_to_load)-1}/{len(pathway_gene_universe)}')
if missing:
    print(f'Missing ({len(missing)}): {missing[:10]}...')

print(f'Loading {len(cols_to_load)} columns...')
new_expr = pl.read_csv(NEW_MATRIX_PATH, columns=cols_to_load, infer_schema_length=100).to_pandas()
if 'sample_name' in new_expr.columns:
    new_expr.set_index('sample_name', inplace=True)
new_expr.fillna(0, inplace=True)

# ── Align with metadata ───────────────────────────────────────────────────────
new_meta = pd.read_csv(NEW_META_PATH)
new_meta_idx = new_meta.set_index('sample_name')
common = new_expr.index.intersection(new_meta_idx.index)
new_expr = new_expr.loc[common]
new_labels = new_meta_idx.loc[common, 'subclass_label']
valid = new_labels.notna()
new_expr, new_labels = new_expr[valid], new_labels[valid]
print(f'Aligned: {new_expr.shape[0]:,} cells, {new_labels.nunique()} cell types')

# ── Preprocess using the same pipeline ───────────────────────────────────────
new_hvg = np.zeros((new_expr.shape[0], len(pathway_gene_universe)), dtype=np.float32)
for i, g in enumerate(pathway_gene_universe):
    if g in new_expr.columns:
        new_hvg[:, i] = new_expr[g].values
del new_expr; gc.collect()

lib = new_hvg.sum(axis=1, keepdims=True)
lib = np.where(lib == 0, 1, lib)
new_X = np.log1p((new_hvg / lib) * 1e4).astype(np.float32)
del new_hvg; gc.collect()
print(f'Preprocessed: {new_X.shape}')

# ── Inference with Unknown detection ─────────────────────────────────────────
model.eval()
new_preds_list, new_probs_list, new_attn_list = [], [], []
UNKNOWN_THRESHOLD = 0.95
BATCH = 256

with torch.no_grad():
    for i in range(0, len(new_X), BATCH):
        xb = torch.from_numpy(new_X[i:i+BATCH]).to(DEVICE)
        logits, attn = model(xb, return_attention=True)
        probs  = F.softmax(logits, dim=-1)
        max_p, preds = probs.max(dim=-1)
        preds[max_p < UNKNOWN_THRESHOLD] = -1   # UNKNOWN
        new_preds_list.append(preds.cpu().numpy())
        new_probs_list.append(max_p.cpu().numpy())
        new_attn_list.append(attn.cpu().numpy())

new_preds = np.concatenate(new_preds_list)
new_probs = np.concatenate(new_probs_list)
new_attn_matrix = np.vstack(new_attn_list)

# Decode predictions (Unknown = -1 stays as 'Unknown')
pred_labels = np.where(
    new_preds == -1,
    'Unknown',
    le.inverse_transform(np.clip(new_preds, 0, n_classes-1))
)

print(f'\nPrediction distribution (incl. Unknown):')
print(pd.Series(pred_labels).value_counts())

n_unknown = (new_preds == -1).sum()
pct_unknown = n_unknown / len(new_preds) * 100
print(f'\nUnknown cells: {n_unknown:,} ({pct_unknown:.1f}%) — these may be novel cell types!')

In [ ]:
# ── Evaluate on cells whose true label is known to the model ─────────────────
known_mask = new_labels.isin(class_names) & (new_preds != -1)
print(f'Cells with known labels (excl. Unknown predictions): {known_mask.sum():,} / {len(new_labels):,}')

if known_mask.sum() > 0:
    true_enc = le.transform(new_labels[known_mask].values)
    pred_enc = new_preds[known_mask.values]
    acc = (true_enc == pred_enc).mean()
    print(f'Transfer Accuracy (known classes): {acc:.4f} ({acc*100:.2f}%)')
    print()
    print(classification_report(true_enc, pred_enc, target_names=class_names, digits=3))

    cm = confusion_matrix(true_enc, pred_enc)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(max(10, n_classes), max(8, n_classes - 2)))
    ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(
        ax=ax, cmap='Blues', colorbar=True, xticks_rotation=45)
    ax.set_title(f'SMART-seq Transfer — Confusion Matrix (Acc: {acc:.3f})')
    plt.tight_layout()
    plt.savefig('fig_smartseq_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

# ── UMAP on SMART-seq attention scores (colored by prediction) ───────────────
MAX_TRANSFER_CELLS = 3000
sub_idx = np.random.choice(len(new_attn_matrix), min(MAX_TRANSFER_CELLS, len(new_attn_matrix)), replace=False)
attn_sub = new_attn_matrix[sub_idx]
labs_sub = pred_labels[sub_idx]

reducer2 = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3, random_state=SEED, metric='cosine')
umap2 = reducer2.fit_transform(attn_sub)

all_label_names = list(class_names) + ['Unknown']
extended_palette = {cls: sns.color_palette('tab20', n_classes)[i] for i, cls in enumerate(class_names)}
extended_palette['Unknown'] = (0.6, 0.6, 0.6)   # gray for unknown

fig, ax = plt.subplots(figsize=(10, 8))
for lbl in all_label_names:
    m = labs_sub == lbl
    if m.sum() == 0: continue
    s = 5 if lbl != 'Unknown' else 2
    ax.scatter(umap2[m, 0], umap2[m, 1], c=[extended_palette[lbl]], s=s,
               alpha=0.8, label=f'{lbl} (n={m.sum()})' if lbl=='Unknown' else lbl)
ax.set(xlabel='UMAP 1', ylabel='UMAP 2',
       title='SMART-seq Transfer — Attention-Score UMAP\n(gray = Unknown / potential novel cell types)')
ax.legend(markerscale=4, fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('fig_transfer_umap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_transfer_umap.png')
print('\n>>> Gray cells = model uncertain. Inspect these clusters for potential novel cell types.')

## 12. Summary

| Metric | Value |
|---|---|
| Test Accuracy | see above |
| Interpretability | ✅ Pathway attention scores per cell |
| Biological grounding | ✅ Reactome pathway mask |
| Unknown cell detection | ✅ Prob threshold = 0.95 |
| UMAP space | ✅ Biological pathway space (not PCA/gene expression) |
| Batch insensitivity | ✅ No batch labels needed |

### Key outputs
- `fig_attention_umap.png` — cells in biological pathway space
- `fig_top_pathways_per_celltype.png` — which Reactome pathways drive each cell type
- `fig_pathway_heatmap.png` — cross-cell-type pathway attention comparison
- `fig_transfer_umap.png` — cross-dataset transfer with unknown cell flagging
- `fig_confusion_matrix.png` — standard test set evaluation

### Next steps
- Increase `embed_dim` from 48 → 96 for deeper representation
- Use TF regulon masks (MSigDB C3) instead of Reactome for developmental data
- Apply `pseudotime` trajectory analysis on the attention-score UMAP using `scanpy.tl.dpt`
- Compare attention scores between conditions (healthy vs. disease) to identify key regulatory pathways